# distributed-sampler-shard — ex1: shard a dataset across ranks with DistributedSampler

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `distributed-sampler-shard`. Running the final beacon cell reports progress against the `Distributed: DistributedSampler shard` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: DistributedSampler shard` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`distributed-sampler-shard`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "distributed-sampler-shard"
DD_SUBTOPIC = "Distributed: DistributedSampler shard"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Distributed: `DistributedSampler` shard — quick refresher

`DistributedSampler` partitions a dataset across `world_size` ranks so each rank sees a DISJOINT slice each epoch:

```python
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

sampler = DistributedSampler(
    dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=0,
)
loader = DataLoader(dataset, batch_size=B, sampler=sampler)

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)   # MUST call so shuffle re-derives
    for batch in loader: ...
```

**What `num_replicas` and `rank` do.** The sampler builds a deterministic permutation of `[0, len(dataset))`, then yields indices `i` where `i % num_replicas == rank`. Rank 0 sees indices 0, W, 2W, ...; rank 1 sees 1, W+1, 2W+1, ... — each rank sees `len(dataset) / num_replicas` items.

**Padding to evenly divide.** If `len(dataset)` isn't a multiple of `num_replicas`, by default the sampler PADS the index list by wrapping from the start, so every rank gets exactly the same number of items. Pass `drop_last=True` to drop the tail instead.

**`set_epoch` is mandatory for shuffled training.** The shuffle uses `epoch + seed` as its RNG state. Without `set_epoch(epoch)`, every epoch re-uses the same permutation — your model sees the same batch order forever.

**Use `sampler=`, NOT `shuffle=True`.** The DataLoader's `shuffle=True` is mutually exclusive with `sampler=`. The shuffle must live inside the sampler so it's coordinated across ranks.

### Exercise 1 — shard a dataset across ranks with DistributedSampler

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `DistributedSampler(dataset, num_replicas, rank)` to produce a disjoint index shard for each rank, and verify the union of shards covers (with padding) the full dataset.
> Keywords: distributed-sampler, sharding, set-epoch, data-parallel
> ```

**KCs targeted:** `distributed-sampler-construction`, `set-epoch-reshuffle`

Implement `ex1_collect_shards(dataset, world_size, seed)`. The from-scratch verification that `DistributedSampler` shards the way you expect.

1. For each `rank` in `range(world_size)`:
   a. Build `DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=seed)`.
   b. Call `sampler.set_epoch(0)` (mandatory for reproducibility).
   c. Iterate the sampler and collect the indices into a list.
2. Return a `list[list[int]]` of length `world_size`, outer index = rank, inner = that rank's epoch-0 index list.

Inputs:
- `dataset`: any `Dataset` (the sampler only uses `len(dataset)`).
- `world_size`: int >= 1.
- `seed`: int.

Output: `list[list[int]]`.

(No multiprocessing required — the sampler is a pure iterator that takes `rank` as a constructor arg.)

In [ ]:
def ex1_collect_shards(dataset, world_size, seed):
    shards = []
    for rank in range(world_size):
        sampler = DistributedSampler(
            dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=seed,
        )
        sampler.set_epoch(0)
        shards.append(list(sampler))
    return shards


<details><summary>Solution</summary>

```python
def ex1_collect_shards(dataset, world_size, seed):
    shards = []
    for rank in range(world_size):
        sampler = DistributedSampler(
            dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=True,
            seed=seed,
        )
        sampler.set_epoch(0)
        shards.append(list(sampler))
    return shards
```

**Why a loop over ranks works without multiprocessing.** `DistributedSampler` doesn't open sockets or call into `torch.distributed` — it's a pure iterator parameterized by `rank` and `num_replicas`. Each rank's sampler is INDEPENDENT; the math (modular striding into a shuffled permutation) is deterministic given the seed and epoch.

**`set_epoch(epoch)` is the contract.** Without it, every call to `iter(sampler)` re-uses the same RNG seed, producing the same permutation epoch after epoch. The training loop must call it before each epoch — typically `for epoch in range(N): sampler.set_epoch(epoch); for batch in loader: ...`.

**Padding is the default for evenly-divisible counts.** If `len(dataset) % world_size != 0`, the sampler wraps the index list from the start so every rank gets exactly `ceil(len / world_size)` items. This matters for `all_reduce` of gradients: every rank must run the same number of forward passes per epoch, or some ranks block waiting for others. Pass `drop_last=True` if you'd rather throw away the tail.

**Pair with `DataLoader(sampler=sampler)`, NOT `shuffle=True`.** The two are mutually exclusive.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()